#### Data Ingestion: Petroleum Consumption (EIA API)

**Latency requirement:** The source (EIA) updates data weekly. For this task, a batch 
load (one-time historical pull) is sufficient – no near-real-time processing is required.

**Data volume:** 6,447 rows for 2009–2026, 
2,149,203 bytes (~2.05 MB) in raw JSON format.


The dataset represents weekly U.S. petroleum product consumption (Product Supplied) 
broken down by product type (gasoline, diesel, jet fuel, etc.), retrieved via the 
official EIA Open Data API.

### Design decisions

- **`requests` over `aiohttp`:** only 2 sequential API calls (pagination) are needed, 
  not many parallel requests to different sources – async would add complexity without benefit.
- **`tenacity` over a manual retry loop:** industry-standard library for retry logic – 
  declarative `@retry` decorator, easier to read/maintain than a hand-rolled loop.
- **Dynamic pagination (`while offset < total`) over hardcoded offsets:** automatically 
  adapts to the real row count, which may change over time, without manual recalculation.
- **`logging` over `print`:** standard practice – supports log levels (INFO/WARNING/ERROR) 
  and integrates more easily with future monitoring/alerting than raw print statements.

In [0]:
dbutils.library.restartPython()

In [0]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("eia_ingestion")

In [0]:
import requests
import json
from tenacity import retry, stop_after_attempt, wait_exponential

API_KEY = dbutils.secrets.get(scope="eia_api", key="eia-api-key")
BASE_URL = "https://api.eia.gov/v2/petroleum/cons/wpsup/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_page(offset, length=5000):
    params ={
        "frequency": "weekly",
        "data[0]": "value",
        "start": "2009-01-01",
        "end": "2026-08-21",
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    return response


def fetch_all_data(page_size=5000):
    all_data = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_page(offset=offset, length=page_size)
        body = response.json()["response"]
        
        if total is None:
            total = int(body["total"])
            logger.info(f"Total rows available: {total}")
        
        all_data.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_data)}/{total} rows so far")

    return all_data, total

In [0]:
all_data, total = fetch_all_data()

total_bytes = len(json.dumps(all_data).encode("utf-8"))
logger.info(f"Total rows fetched: {len(all_data)}")
logger.info(f"Total raw JSON size: {total_bytes} bytes ({total_bytes / 1024:.1f} KB)")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev_ua5816bd.roksolana_shendiu770.raw_files;

In [0]:
output_path = "/Volumes/dbr_dev_ua5816bd/roksolana_shendiu770/raw_files/petroleum_raw.json"

with open(output_path, "w") as f:
    json.dump(all_data, f)

logger.info(f"Saved {len(all_data)} rows to {output_path}")


In [0]:
imported_df = spark.read.json(output_path)
imported_df.printSchema()

In [0]:
imported_df.show(5)